In [ ]:
import pandas as pd
import numpy as np

#setting the random seed
np.random.seed(42)

#defining scale parameters
n_customers = 5000
n_transactions = 25000

#customer profile setup
print("Generating realistic M-pesa customer demographic profiles...")
customer_ids = [f"CUST_{str(i).zfill(5)}" for i in range(1,n_customers +  1)]

signup_dates = pd.date_range(start="2023-01-01",end = "2025-12-31",periods=n_customers)

#building the customer dataframe
customers_df = pd.DataFrame({
  "customer_id":customer_ids,
  "signup_date":signup_dates,
  "age":np.random.randint(18,70,size = n_customers),
  "region":np.random.choice(["Nairobi","Mombasa","Kisumu","Nakuru","Eldoret"],size=n_customers,p=[0.4,0.2,0.15,0.15,0.1]),
  "kyc_tier":np.random.choice(["Tier_1","Tier_2","Tier_3"],size=n_customers,p=[0.2,0.5,0.3]),
  "is_active_agent_network":np.random.choice([0,1],size=n_customers,p=[0.7,0.3])

})

#transaction logs set up
print("Generating M-pesa transaction logs (peer,paybill,till)...")
tx_customer_ids = np.random.choice(customer_ids,size=n_transactions)
tx_timestamps = pd.date_range(start = "2026-01-01",end="2026-09-26",periods=n_transactions)

#building the transaction dataframe
transactions_df = pd.DataFrame({
    "transaction_id": [f"TXN_{str(i).zfill(7)}" for i in range(1, n_transactions + 1)],
    "customer_id":tx_customer_ids,
    "timestamp":tx_timestamps,
    "amount_kes":np.random.exponential(scale=2500,size=n_transactions).round(2),
    "transaction_type":np.random.choice(["p2p_send","paybill","buy_goods_till","cash_withdrawal"],size=n_transactions,p=[0.45,0.25,0.20,0.10]),
    "fee_kes":lambda df:df['amount_kes'] * 0.015
}) 

transactions_df["fee_kes"] = (transactions_df["amount_kes"] * 0.012).round(2)

#defining churn-the target variable
active_counts = transactions_df.groupby("customer_id").size()
churned_customers = active_counts[active_counts < 3].index.tolist()
customers_df["churn"] = customers_df["customer_id"].isin(churned_customers).astype(int)

#saving and displaying results
customers_df.to_csv("data/raw/customer_demographics.csv",index = False)
transactions_df.to_csv("data/raw/mpesa_transactions.csv",index = False)

print("✅ Phase 1 Data Ingestion Complete!")
print(f"Saved {len(customers_df)} customer records and {len(transactions_df)} transaction logs.")
display(customers_df.head(3))
display(transactions_df.head(3))

In [ ]:
#setting up the visual style
import matplotlib.pyplot as plt
import seaborn as sns

#setting style for clean professional plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10,5)

#customer churn distribution(bar plot)
plt.figure(figsize = (6,4))
ax = sns.countplot(data=customers_df,x="churn",palette = "viridis")

plt.title("Customer Churn Distribution(0=Retained,1=Churned)",fontsize=12,fontweight='bold')
plt.xlabel("Churn Status")
plt.ylabel("Customer Count")

for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

    plt.show()


#mpesa transaction volumes by type(horizontal bar plot)
plt.figure(figsize=(9, 4))
sns.countplot(data=transactions_df, y="transaction_type", order=transactions_df['transaction_type'].value_counts().index, palette="mako")
plt.title("M-Pesa Transaction Volumes by Type (KES Ecosystem)", fontsize=12, fontweight='bold')
plt.xlabel("Total Transaction Count")
plt.ylabel("Transaction Type")
plt.show()
print(f"📊 Quick Insight: Overall Churn Rate in generated cohort is {(customers_df['churn'].mean() * 100):.2f}%")
    